In [1]:
from getpass import getpass
import os

# Paste your OpenAI API key in the secure prompt at the top of VS Code
OPENAI_API_KEY = getpass("Enter your OpenAI API key: ").strip()

if not OPENAI_API_KEY:
  raise ValueError("API key cannot be empty.")

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
print("API key configured successfully for this session!")

API key configured successfully for this session!


In [4]:
from typing import List
from fastapi import FastAPI, File, HTTPException, UploadFile, status
from pydantic import BaseModel
import nest_asyncio
import os
import uvicorn

# LangChain Imports
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

nest_asyncio.apply()

# Initialize FastAPI app here
app = FastAPI(title="RAG Document Processing API", version="1.0.0")

FAISS_INDEX_PATH = "faiss_index"
UPLOAD_DIR = "uploaded_docs"
os.makedirs(UPLOAD_DIR, exist_ok=True)
os.makedirs(FAISS_INDEX_PATH, exist_ok=True)

ACTIVE_API_KEY = os.getenv("OPENAI_API_KEY")
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small", openai_api_key=ACTIVE_API_KEY
)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)


class QueryRequest(BaseModel):
  question: str
  top_k: int = 4


class QueryResponse(BaseModel):
  answer: str
  sources: List[str]


@app.post("/upload", status_code=status.HTTP_201_CREATED)
async def upload_document(file: UploadFile = File(...)):
  file_ext = os.path.splitext(file.filename)[1].lower()
  file_path = os.path.join(UPLOAD_DIR, file.filename)

  content = await file.read()
  with open(file_path, "wb") as buffer:
    buffer.write(content)

  if file_ext == ".pdf":
    loader = PyPDFLoader(file_path)
  elif file_ext == ".txt":
    loader = TextLoader(file_path, encoding="utf-8")
  else:
    raise HTTPException(status_code=400, detail="Only .pdf and .txt files allowed.")

  docs = loader.load()
  for doc in docs:
    doc.metadata.update({"source_file": file.filename})

  chunks = text_splitter.split_documents(docs)

  if os.path.exists(os.path.join(FAISS_INDEX_PATH, "index.faiss")):
    vector_store = FAISS.load_local(
        FAISS_INDEX_PATH, embeddings, allow_dangerous_deserialization=True
    )
    vector_store.add_documents(chunks)
  else:
    vector_store = FAISS.from_documents(chunks, embeddings)

  vector_store.save_local(FAISS_INDEX_PATH)
  return {
      "filename": file.filename,
      "status": "Success",
      "chunks_indexed": len(chunks),
  }


@app.post("/query", response_model=QueryResponse)
async def query_rag(request: QueryRequest):
  vector_store = FAISS.load_local(
      FAISS_INDEX_PATH, embeddings, allow_dangerous_deserialization=True
  )
  retriever = vector_store.as_retriever(search_kwargs={"k": request.top_k})
  retrieved_docs = retriever.invoke(request.question)

  context = "\n\n".join([doc.page_content for doc in retrieved_docs])
  sources = list(
      set([doc.metadata.get("source_file", "Unknown") for doc in retrieved_docs])
  )

  llm = ChatOpenAI(
      model="gpt-4o-mini", temperature=0.2, openai_api_key=ACTIVE_API_KEY
  )
  prompt_template = ChatPromptTemplate.from_messages([
      (
          "system",
          (
              "Answer strictly using the provided context. If unknown, state"
              " 'I cannot find the answer.'"
          ),
      ),
      ("human", "Context:\n{context}\n\nQuestion: {question}"),
  ])

  chain = prompt_template | llm
  response = chain.invoke({"context": context, "question": request.question})
  return QueryResponse(answer=response.content, sources=sources)


@app.get("/health")
def health_check():
  return {"status": "healthy"}


print("FastAPI app code loaded successfully!")

FastAPI app code loaded successfully!


In [ ]:
import threading
import uvicorn


def run_server():
  uvicorn.run(app, host="127.0.0.1", port=8002, log_level="info")


server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
print("FastAPI server started in the background on port 8002!")

FastAPI server started in the background on port 8002!


INFO:     Started server process [14360]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8002 (Press CTRL+C to quit)


INFO:     127.0.0.1:56516 - "GET / HTTP/1.1" 404 Not Found
INFO:     127.0.0.1:56516 - "GET /favicon.ico HTTP/1.1" 404 Not Found
INFO:     127.0.0.1:64263 - "GET /docs HTTP/1.1" 200 OK
INFO:     127.0.0.1:64263 - "GET /openapi.json HTTP/1.1" 200 OK
INFO:     127.0.0.1:51402 - "POST /upload HTTP/1.1" 500 Internal Server Error


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "c:\Users\prati\AppData\Local\Programs\Python\Python313\Lib\site-packages\uvicorn\protocols\http\httptools_impl.py", line 422, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        self.scope, self.receive, self.send
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "c:\Users\prati\AppData\Local\Programs\Python\Python313\Lib\site-packages\uvicorn\middleware\proxy_headers.py", line 63, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\prati\AppData\Local\Programs\Python\Python313\Lib\site-packages\fastapi\applications.py", line 1194, in __call__
    await super().__call__(scope, receive, send)
  File "c:\Users\prati\AppData\Local\Programs\Python\Python313\Lib\site-packages\starlette\applications.py", line 96, in __call__
    await self

In [6]:
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import seaborn as sns

# Initialize LLM
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, openai_api_key=ACTIVE_API_KEY)


# Define Custom Tools
@tool
def calculate_summary_statistics(file_path: str) -> str:
  """Loads a CSV file and calculates basic descriptive statistics and missing values."""
  try:
    df = pd.read_csv(file_path)
    buffer = [
        f"Dataset Shape: {df.shape}",
        f"Columns & Types:\n{df.dtypes.to_string()}",
        f"Missing Values:\n{df.isnull().sum().to_string()}",
        f"Summary Statistics:\n{df.describe().to_string()}",
    ]
    return "\n\n".join(buffer)
  except Exception as e:
    return f"Error processing file: {str(e)}"


@tool
def generate_correlation_heatmap(
    file_path: str, save_path: str = "uploaded_docs/heatmap.png"
) -> str:
  """Generates a Seaborn correlation heatmap and saves it as an image file."""
  try:
    df = pd.read_csv(file_path)
    numeric_df = df.select_dtypes(include=[np.number])

    plt.figure(figsize=(8, 6))
    sns.heatmap(
        numeric_df.corr(), annot=True, cmap="coolwarm", fmt=".2f", linewidths=0.5
    )
    plt.title("Correlation Heatmap")
    plt.tight_layout()

    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    plt.savefig(save_path)
    plt.close()
    return f"Successfully generated and saved correlation heatmap to {save_path}"
  except Exception as e:
    return f"Error generating heatmap: {str(e)}"


tools = [calculate_summary_statistics, generate_correlation_heatmap]

prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        (
            "You are an expert Data Analyst AI Agent. Use your available tools"
            " to analyze datasets and generate charts."
        ),
    ),
    ("placeholder", "{chat_history}"),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
])

agent = create_tool_calling_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

print("Data Analyst Agent is fully configured and ready!")

ImportError: cannot import name 'AgentExecutor' from 'langchain.agents' (c:\Users\prati\AppData\Local\Programs\Python\Python313\Lib\site-packages\langchain\agents\__init__.py)